# CTE-Net ablation: statistical comparisons across ten seeds

This notebook compares the complete **CTE-Net** model with the two ablation variants:

- **Without TE Module**
- **Without Transformer**

It reconstructs all window-level metrics from the saved out-of-fold probabilities, rather than relying on manually copied summaries. Two complementary analyses are performed:

1. **Exploratory seed-level analysis:** paired two-sided Wilcoxon signed-rank tests across the ten matched training seeds, with Holm correction across the six prespecified comparisons (three metrics × two ablations).
2. **Primary participant-level analysis:** exact paired McNemar tests based on one consensus decision per participant and model, with Holm correction across the two ablation comparisons.

Random seeds quantify optimization variability and are **not** treated as independent clinical observations. The participant-level analysis treats the participant as the paired unit.

In [9]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

from IPython.display import display
from scipy.stats import binomtest, wilcoxon
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)


# ============================================================
# 1. PATHS AND PRESPECIFIED ANALYSES
# ============================================================

INPUT_ROOTS = {
    "CTE-Net": Path(
        "/kaggle/input/datasets/alejandragomezr/results-models-c-tenet/"
        "C-TENet/HybridTransformerTEKTE_reinference_complete"
    ),
    "Without TE Module": Path(
        "/kaggle/input/datasets/alejandragomezr/results-ablation/"
        "Ablation without TE"
    ),
    "Without Transformer": Path(
        "/kaggle/input/datasets/alejandragomezr/results-ablation/"
        "Ablation without Trans"
    ),
}

EXPECTED_FILES = {
    "CTE-Net": "CTE_Net_all_window_predictions.csv",
    "Without TE Module": "WithoutTE_all_window_predictions.csv",
    "Without Transformer": "WithoutTransformer_all_window_predictions.csv",
}

OUTPUT_ROOT = Path("/kaggle/working/ablation_statistical_tests")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

EXPECTED_SEEDS = list(range(10))
EXPECTED_FOLDS_PER_SEED = 5
EXPECTED_SUBJECTS = 120

# These are the three metrics currently reported in the ablation table.
# Holm correction is applied jointly across 3 metrics × 2 ablations = 6 tests.
PRIMARY_WILCOXON_METRICS = [
    "accuracy",
    "sensitivity",
    "precision",
]

ALL_METRICS = [
    "accuracy",
    "balanced_accuracy",
    "sensitivity",
    "specificity",
    "precision",
    "f1_score",
    "roc_auc",
]

ABLATIONS = ["Without TE Module", "Without Transformer"]

print("Output directory:", OUTPUT_ROOT)

Output directory: /kaggle/working/ablation_statistical_tests


In [10]:
# ============================================================
# 2. FILE DISCOVERY AND SCHEMA NORMALIZATION
# ============================================================

ALIASES = {
    "seed": ["seed", "base_seed", "repeat_id"],
    "fold": ["fold", "fold_id", "test_fold"],
    "subject_id": ["subject_id", "subject", "participant_id", "participant"],
    "y_true": ["y_true", "label", "true_label", "target"],
    "prob_adhd": [
        "prob_adhd",
        "y_prob",
        "probability",
        "predicted_probability",
        "adhd_probability",
    ],
    "window_id": ["window_id", "window_index", "window_number", "window"],
}


def locate_prediction_file(model_name, root, expected_name):
    if not root.exists():
        raise FileNotFoundError(
            f"The input directory for {model_name} does not exist:\n{root}"
        )

    exact_matches = sorted(root.rglob(expected_name))
    if len(exact_matches) == 1:
        return exact_matches[0]
    if len(exact_matches) > 1:
        raise RuntimeError(
            f"More than one {expected_name} file was found for {model_name}:\n"
            + "\n".join(str(path) for path in exact_matches)
        )

    fallback = sorted(root.rglob("*all_window_predictions.csv"))
    if len(fallback) == 1:
        print(
            f"WARNING: {expected_name} was not found for {model_name}; "
            f"using {fallback[0].name}."
        )
        return fallback[0]

    raise FileNotFoundError(
        f"Could not uniquely locate the window predictions for {model_name} in:\n"
        f"{root}\nCandidates: {[str(path) for path in fallback]}"
    )


def choose_column(frame, canonical_name, required=True):
    lower_to_original = {str(column).lower(): column for column in frame.columns}
    for candidate in ALIASES[canonical_name]:
        if candidate.lower() in lower_to_original:
            return lower_to_original[candidate.lower()]
    if required:
        raise KeyError(
            f"No column for '{canonical_name}' was found. "
            f"Available columns: {list(frame.columns)}"
        )
    return None


def integer_from_value(value, field_name):
    if pd.isna(value):
        raise ValueError(f"Missing value in {field_name}.")
    if isinstance(value, (int, np.integer)):
        return int(value)
    if isinstance(value, (float, np.floating)) and float(value).is_integer():
        return int(value)
    match = re.search(r"-?\\d+", str(value))
    if match is None:
        raise ValueError(f"Could not parse {field_name}={value!r} as an integer.")
    return int(match.group())


def canonical_subject_id(value):
    text = str(value).strip().lower()
    if text.endswith(".mat"):
        text = text[:-4]
    return text


def normalize_window_predictions(path, model_name):
    raw = pd.read_csv(path)

    selected = {
        name: choose_column(raw, name, required=(name != "window_id"))
        for name in ALIASES
    }

    normalized = pd.DataFrame(
        {
            "model": model_name,
            "seed": raw[selected["seed"]].map(
                lambda value: integer_from_value(value, "seed")
            ),
            "fold": raw[selected["fold"]].map(
                lambda value: integer_from_value(value, "fold")
            ),
            "subject_id": raw[selected["subject_id"]].map(canonical_subject_id),
            "y_true": pd.to_numeric(raw[selected["y_true"]], errors="raise").astype(int),
            "prob_adhd": pd.to_numeric(
                raw[selected["prob_adhd"]], errors="raise"
            ).astype(float),
        }
    )

    if selected["window_id"] is None:
        normalized["window_id"] = normalized.groupby(
            ["seed", "fold", "subject_id"], sort=False
        ).cumcount()
    else:
        normalized["window_id"] = raw[selected["window_id"]].astype(str)

    if not np.isin(normalized["y_true"], [0, 1]).all():
        raise ValueError(f"{model_name} contains labels other than 0 and 1.")
    if not np.isfinite(normalized["prob_adhd"]).all():
        raise ValueError(f"{model_name} contains nonfinite probabilities.")
    if not normalized["prob_adhd"].between(0.0, 1.0).all():
        raise ValueError(f"{model_name} contains probabilities outside [0, 1].")

    duplicate_key = ["seed", "fold", "subject_id", "window_id"]
    duplicated = normalized.duplicated(duplicate_key, keep=False)
    if duplicated.any():
        raise RuntimeError(
            f"{model_name} contains duplicate window predictions for the same "
            "seed/fold/participant/window."
        )

    return normalized.sort_values(duplicate_key).reset_index(drop=True)


input_paths = {}
normalized_frames = []

for model_name, root in INPUT_ROOTS.items():
    path = locate_prediction_file(
        model_name=model_name,
        root=root,
        expected_name=EXPECTED_FILES[model_name],
    )
    input_paths[model_name] = path
    normalized_frames.append(normalize_window_predictions(path, model_name))

window_predictions = pd.concat(normalized_frames, ignore_index=True)

print("Prediction files used:")
for model_name, path in input_paths.items():
    print(f"- {model_name}: {path}")

display(
    window_predictions.groupby("model").agg(
        rows=("prob_adhd", "size"),
        seeds=("seed", "nunique"),
        folds=("fold", "nunique"),
        subjects=("subject_id", "nunique"),
    )
)

Prediction files used:
- CTE-Net: /kaggle/input/datasets/alejandragomezr/results-models-c-tenet/C-TENet/HybridTransformerTEKTE_reinference_complete/CTE_Net_all_window_predictions.csv
- Without TE Module: /kaggle/input/datasets/alejandragomezr/results-ablation/Ablation without TE/ablation_without_te_repeated_10seeds/WithoutTE_all_window_predictions.csv
- Without Transformer: /kaggle/input/datasets/alejandragomezr/results-ablation/Ablation without Trans/ablation_without_transformer_repeated_10seeds/WithoutTransformer_all_window_predictions.csv


,rows,seeds,folds,subjects
model,,,,
CTE-Net,82130,10,5,120
Without TE Module,82130,10,5,120
Without Transformer,82130,10,5,120


In [11]:
# ============================================================
# 3. PROTOCOL AUDIT: SEEDS, FOLDS, PARTICIPANTS AND PARTITIONS
# ============================================================

audit_rows = []

for model_name, group in window_predictions.groupby("model", sort=False):
    seeds = sorted(group["seed"].unique().tolist())
    if seeds != EXPECTED_SEEDS:
        raise RuntimeError(
            f"{model_name}: expected seeds {EXPECTED_SEEDS}, obtained {seeds}."
        )

    fold_counts = group.groupby("seed")["fold"].nunique()
    if not (fold_counts == EXPECTED_FOLDS_PER_SEED).all():
        raise RuntimeError(
            f"{model_name}: at least one seed does not contain "
            f"{EXPECTED_FOLDS_PER_SEED} folds:\n{fold_counts}"
        )

    subject_counts = group.groupby("seed")["subject_id"].nunique()
    if not (subject_counts == EXPECTED_SUBJECTS).all():
        raise RuntimeError(
            f"{model_name}: at least one seed does not contain "
            f"{EXPECTED_SUBJECTS} participants:\n{subject_counts}"
        )

    label_counts = group.groupby(["seed", "subject_id"])["y_true"].nunique()
    if not (label_counts == 1).all():
        raise RuntimeError(f"{model_name}: inconsistent labels were found.")

    subject_fold_counts = group.groupby(
        ["seed", "subject_id"]
    )["fold"].nunique()
    if not (subject_fold_counts == 1).all():
        raise RuntimeError(
            f"{model_name}: a participant appears in multiple test folds "
            "within the same seed."
        )

    audit_rows.append(
        {
            "model": model_name,
            "n_rows": len(group),
            "n_seeds": group["seed"].nunique(),
            "folds_per_seed_min": int(fold_counts.min()),
            "folds_per_seed_max": int(fold_counts.max()),
            "subjects_per_seed_min": int(subject_counts.min()),
            "subjects_per_seed_max": int(subject_counts.max()),
        }
    )

# Confirm identical subject-wise test partitions across the three models.
reference = window_predictions.loc[
    window_predictions["model"] == "CTE-Net",
    ["seed", "fold", "subject_id", "y_true"],
].drop_duplicates()

for ablation_name in ABLATIONS:
    comparison = window_predictions.loc[
        window_predictions["model"] == ablation_name,
        ["seed", "fold", "subject_id", "y_true"],
    ].drop_duplicates()

    paired = reference.merge(
        comparison,
        on=["seed", "fold", "subject_id"],
        how="outer",
        suffixes=("_cte", "_ablation"),
        indicator=True,
        validate="one_to_one",
    )
    if not (paired["_merge"] == "both").all():
        raise RuntimeError(
            f"The subject-wise test partitions differ between CTE-Net and "
            f"{ablation_name}."
        )
    if not np.array_equal(
        paired["y_true_cte"].to_numpy(),
        paired["y_true_ablation"].to_numpy(),
    ):
        raise RuntimeError(
            f"Participant labels differ between CTE-Net and {ablation_name}."
        )

audit = pd.DataFrame(audit_rows)
audit["partitions_match_cte_net"] = True
audit.to_csv(OUTPUT_ROOT / "Ablation_input_protocol_audit.csv", index=False)

print("Protocol audit passed.")
display(audit)

Protocol audit passed.


,model,n_rows,n_seeds,folds_per_seed_min,folds_per_seed_max,subjects_per_seed_min,subjects_per_seed_max,partitions_match_cte_net
0,CTE-Net,82130,10,5,5,120,120,True
1,Without TE Module,82130,10,5,5,120,120,True
2,Without Transformer,82130,10,5,5,120,120,True


In [12]:
# ============================================================
# 4. RECOMPUTE WINDOW METRICS BY FOLD AND SEED
# ============================================================

def calculate_binary_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)
    y_pred = (y_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()
    specificity = tn / (tn + fp) if (tn + fp) else np.nan

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "sensitivity": recall_score(
            y_true, y_pred, pos_label=1, zero_division=0
        ),
        "specificity": specificity,
        "precision": precision_score(
            y_true, y_pred, pos_label=1, zero_division=0
        ),
        "f1_score": f1_score(
            y_true, y_pred, pos_label=1, zero_division=0
        ),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


fold_rows = []
for (model_name, seed, fold), group in window_predictions.groupby(
    ["model", "seed", "fold"], sort=True
):
    fold_rows.append(
        {
            "model": model_name,
            "seed": int(seed),
            "fold": int(fold),
            "n_windows": int(len(group)),
            **calculate_binary_metrics(group["y_true"], group["prob_adhd"]),
        }
    )

window_metrics_by_fold = pd.DataFrame(fold_rows)

# Exactly the principal protocol: average the five test folds within each seed.
window_metrics_by_seed = (
    window_metrics_by_fold
    .groupby(["model", "seed"], as_index=False)[ALL_METRICS]
    .mean()
)

summary_rows = []
for model_name, group in window_metrics_by_seed.groupby("model", sort=False):
    for metric in ALL_METRICS:
        values = group[metric].to_numpy(dtype=float)
        summary_rows.append(
            {
                "model": model_name,
                "metric": metric,
                "mean": values.mean(),
                "std_sample": values.std(ddof=1),
                "mean_percent": 100 * values.mean(),
                "std_sample_percent": 100 * values.std(ddof=1),
                "n_seeds": len(values),
            }
        )

window_metrics_summary = pd.DataFrame(summary_rows)
window_metrics_summary["Result (%)"] = window_metrics_summary.apply(
    lambda row: (
        f"{row['mean_percent']:.1f} $\\pm$ "
        f"{row['std_sample_percent']:.1f}"
    ),
    axis=1,
)

window_metrics_by_fold.to_csv(
    OUTPUT_ROOT / "Ablation_window_metrics_by_fold_recomputed.csv",
    index=False,
)
window_metrics_by_seed.to_csv(
    OUTPUT_ROOT / "Ablation_window_metrics_by_seed_recomputed.csv",
    index=False,
)
window_metrics_summary.to_csv(
    OUTPUT_ROOT / "Ablation_window_metrics_mean_std.csv",
    index=False,
)

display(
    window_metrics_summary.pivot(
        index="model", columns="metric", values="Result (%)"
    )[ALL_METRICS]
)

metric,accuracy,balanced_accuracy,sensitivity,specificity,precision,f1_score,roc_auc
model,,,,,,,
CTE-Net,80.9 $\pm$ 1.7,80.6 $\pm$ 1.8,84.2 $\pm$ 2.3,77.0 $\pm$ 3.2,82.7 $\pm$ 2.1,82.9 $\pm$ 1.5,87.9 $\pm$ 1.5
Without TE Module,78.7 $\pm$ 0.8,78.3 $\pm$ 0.9,82.3 $\pm$ 2.8,74.2 $\pm$ 3.4,80.8 $\pm$ 1.6,81.0 $\pm$ 1.0,84.8 $\pm$ 1.5
Without Transformer,73.4 $\pm$ 2.2,72.9 $\pm$ 2.3,78.0 $\pm$ 2.8,67.8 $\pm$ 4.9,75.7 $\pm$ 2.5,76.3 $\pm$ 1.9,80.0 $\pm$ 2.1


In [13]:
# ============================================================
# 5. PAIRED WILCOXON TESTS ACROSS THE TEN MATCHED SEEDS
# ============================================================

def holm_adjust(p_values):
    p_values = np.asarray(p_values, dtype=float)
    m = len(p_values)
    order = np.argsort(p_values)
    ordered = p_values[order]
    adjusted_ordered = np.maximum.accumulate(
        (m - np.arange(m)) * ordered
    )
    adjusted_ordered = np.minimum(adjusted_ordered, 1.0)
    adjusted = np.empty(m, dtype=float)
    adjusted[order] = adjusted_ordered
    return adjusted


def paired_seed_values(seed_frame, model_name, metric):
    selected = (
        seed_frame.loc[seed_frame["model"] == model_name, ["seed", metric]]
        .sort_values("seed")
        .set_index("seed")[metric]
    )
    if selected.index.tolist() != EXPECTED_SEEDS:
        raise RuntimeError(
            f"{model_name}/{metric} is not available for the ten expected seeds."
        )
    return selected


wilcoxon_rows = []

for metric in PRIMARY_WILCOXON_METRICS:
    cte_values = paired_seed_values(
        window_metrics_by_seed, "CTE-Net", metric
    )
    for ablation_name in ABLATIONS:
        ablation_values = paired_seed_values(
            window_metrics_by_seed, ablation_name, metric
        )

        differences = cte_values.to_numpy() - ablation_values.to_numpy()
        if np.allclose(differences, 0.0):
            statistic, p_raw = 0.0, 1.0
        else:
            test = wilcoxon(
                cte_values.to_numpy(),
                ablation_values.to_numpy(),
                alternative="two-sided",
                zero_method="wilcox",
                method="auto",
            )
            statistic, p_raw = float(test.statistic), float(test.pvalue)

        wilcoxon_rows.append(
            {
                "metric": metric,
                "comparison": f"CTE-Net vs. {ablation_name}",
                "ablation": ablation_name,
                "n_paired_seeds": len(cte_values),
                "cte_net_mean_percent": 100 * cte_values.mean(),
                "ablation_mean_percent": 100 * ablation_values.mean(),
                "mean_difference_cte_minus_ablation_pp": 100 * differences.mean(),
                "wilcoxon_statistic": statistic,
                "p_raw": p_raw,
            }
        )

wilcoxon_holm = pd.DataFrame(wilcoxon_rows)
wilcoxon_holm["p_holm"] = holm_adjust(wilcoxon_holm["p_raw"])
wilcoxon_holm["significant_holm_0_05"] = wilcoxon_holm["p_holm"] < 0.05
wilcoxon_holm["direction"] = np.where(
    wilcoxon_holm["significant_holm_0_05"],
    np.where(
        wilcoxon_holm["mean_difference_cte_minus_ablation_pp"] > 0,
        "CTE-Net > ablation",
        "CTE-Net < ablation",
    ),
    "No significant difference",
)
wilcoxon_holm["correction_family"] = (
    "Holm across 6 prespecified tests: 3 metrics x 2 ablations"
)
wilcoxon_holm["analysis_unit"] = "random seed (computational repetition)"

wilcoxon_holm.to_csv(
    OUTPUT_ROOT / "Ablation_Wilcoxon_Holm_primary_metrics.csv",
    index=False,
)

print(
    "Exploratory seed-level Wilcoxon tests. "
    "Holm correction family: six comparisons."
)
display(wilcoxon_holm)

Exploratory seed-level Wilcoxon tests. Holm correction family: six comparisons.


,metric,comparison,ablation,n_paired_seeds,cte_net_mean_percent,ablation_mean_percent,mean_difference_cte_minus_ablation_pp,wilcoxon_statistic,p_raw,p_holm,significant_holm_0_05,direction,correction_family,analysis_unit
0,accuracy,CTE-Net vs. Without TE Module,Without TE Module,10,80.908972,78.657716,2.251256,4.0,0.013672,0.041016,True,CTE-Net > ablation,Holm across 6 prespecified tests: 3 metrics x ...,random seed (computational repetition)
1,accuracy,CTE-Net vs. Without Transformer,Without Transformer,10,80.908972,73.393470,7.515502,0.0,0.001953,0.011719,True,CTE-Net > ablation,Holm across 6 prespecified tests: 3 metrics x ...,random seed (computational repetition)
2,sensitivity,CTE-Net vs. Without TE Module,Without TE Module,10,84.186879,82.343345,1.843534,14.0,0.193359,0.193359,False,No significant difference,Holm across 6 prespecified tests: 3 metrics x ...,random seed (computational repetition)
3,sensitivity,CTE-Net vs. Without Transformer,Without Transformer,10,84.186879,77.970961,6.215919,0.0,0.001953,0.011719,True,CTE-Net > ablation,Holm across 6 prespecified tests: 3 metrics x ...,random seed (computational repetition)
4,precision,CTE-Net vs. Without TE Module,Without TE Module,10,82.659138,80.815095,1.844042,9.0,0.064453,0.128906,False,No significant difference,Holm across 6 prespecified tests: 3 metrics x ...,random seed (computational repetition)
5,precision,CTE-Net vs. Without Transformer,Without Transformer,10,82.659138,75.691167,6.967971,0.0,0.001953,0.011719,True,CTE-Net > ablation,Holm across 6 prespecified tests: 3 metrics x ...,random seed (computational repetition)


In [14]:
# ============================================================
# 6. PARTICIPANT CONSENSUS PREDICTIONS
# ============================================================

# First aggregate windows within participant and seed.
participant_predictions_by_seed = (
    window_predictions
    .groupby(["model", "seed", "subject_id"], as_index=False)
    .agg(
        fold=("fold", "first"),
        y_true=("y_true", "first"),
        prob_adhd=("prob_adhd", "mean"),
        n_windows=("window_id", "size"),
    )
)
participant_predictions_by_seed["y_pred"] = (
    participant_predictions_by_seed["prob_adhd"] >= 0.5
).astype(int)
participant_predictions_by_seed["correct"] = (
    participant_predictions_by_seed["y_true"]
    == participant_predictions_by_seed["y_pred"]
).astype(int)

counts = participant_predictions_by_seed.groupby(
    ["model", "seed"]
)["subject_id"].nunique()
if not (counts == EXPECTED_SUBJECTS).all():
    raise RuntimeError(
        "At least one model/seed does not contain the 120 participants."
    )

# Then average each participant's probabilities across the ten seeds.
participant_consensus = (
    participant_predictions_by_seed
    .groupby(["model", "subject_id"], as_index=False)
    .agg(
        y_true=("y_true", "first"),
        prob_adhd=("prob_adhd", "mean"),
        n_seeds=("seed", "nunique"),
    )
)
participant_consensus["y_pred"] = (
    participant_consensus["prob_adhd"] >= 0.5
).astype(int)
participant_consensus["correct"] = (
    participant_consensus["y_true"] == participant_consensus["y_pred"]
).astype(int)

if not (participant_consensus["n_seeds"] == len(EXPECTED_SEEDS)).all():
    raise RuntimeError(
        "At least one participant does not have predictions from all ten seeds."
    )

# Descriptive participant-level metrics for each seed.
participant_metric_rows = []
for (model_name, seed), group in participant_predictions_by_seed.groupby(
    ["model", "seed"], sort=True
):
    participant_metric_rows.append(
        {
            "model": model_name,
            "seed": int(seed),
            "n_subjects": len(group),
            **calculate_binary_metrics(group["y_true"], group["prob_adhd"]),
        }
    )

participant_metrics_by_seed = pd.DataFrame(participant_metric_rows)

participant_summary_rows = []
for model_name, group in participant_metrics_by_seed.groupby("model"):
    for metric in ALL_METRICS:
        values = group[metric].to_numpy(dtype=float)
        participant_summary_rows.append(
            {
                "model": model_name,
                "metric": metric,
                "mean_percent": 100 * values.mean(),
                "std_sample_percent": 100 * values.std(ddof=1),
                "n_seeds": len(values),
            }
        )

participant_metrics_summary = pd.DataFrame(participant_summary_rows)
participant_metrics_summary["Result (%)"] = participant_metrics_summary.apply(
    lambda row: (
        f"{row['mean_percent']:.1f} $\\pm$ "
        f"{row['std_sample_percent']:.1f}"
    ),
    axis=1,
)

participant_predictions_by_seed.to_csv(
    OUTPUT_ROOT / "Ablation_participant_predictions_by_seed.csv",
    index=False,
)
participant_consensus.to_csv(
    OUTPUT_ROOT / "Ablation_participant_consensus_predictions.csv",
    index=False,
)
participant_metrics_by_seed.to_csv(
    OUTPUT_ROOT / "Ablation_participant_metrics_by_seed.csv",
    index=False,
)
participant_metrics_summary.to_csv(
    OUTPUT_ROOT / "Ablation_participant_metrics_mean_std.csv",
    index=False,
)

display(
    participant_metrics_summary.pivot(
        index="model", columns="metric", values="Result (%)"
    )[ALL_METRICS]
)

metric,accuracy,balanced_accuracy,sensitivity,specificity,precision,f1_score,roc_auc
model,,,,,,,
CTE-Net,83.4 $\pm$ 1.9,83.4 $\pm$ 1.9,86.0 $\pm$ 3.1,80.8 $\pm$ 3.4,81.8 $\pm$ 2.5,83.8 $\pm$ 1.9,90.2 $\pm$ 1.6
Without TE Module,82.1 $\pm$ 2.1,82.1 $\pm$ 2.1,87.5 $\pm$ 2.9,76.7 $\pm$ 4.0,79.0 $\pm$ 2.7,83.0 $\pm$ 1.8,87.2 $\pm$ 1.9
Without Transformer,76.7 $\pm$ 3.6,76.7 $\pm$ 3.6,83.8 $\pm$ 3.1,69.5 $\pm$ 6.6,73.5 $\pm$ 4.2,78.3 $\pm$ 3.0,83.4 $\pm$ 2.5


In [15]:
# ============================================================
# 7. EXACT PARTICIPANT-LEVEL MCNEMAR TESTS WITH HOLM CORRECTION
# ============================================================

def exact_mcnemar_against_cte(consensus_frame, ablation_name):
    cte = consensus_frame.loc[
        consensus_frame["model"] == "CTE-Net",
        ["subject_id", "y_true", "y_pred"],
    ]
    ablation = consensus_frame.loc[
        consensus_frame["model"] == ablation_name,
        ["subject_id", "y_true", "y_pred"],
    ]

    paired = cte.merge(
        ablation,
        on="subject_id",
        how="inner",
        suffixes=("_cte", "_ablation"),
        validate="one_to_one",
    )
    if len(paired) != EXPECTED_SUBJECTS:
        raise RuntimeError(
            f"CTE-Net and {ablation_name} do not share all 120 participants."
        )
    if not np.array_equal(
        paired["y_true_cte"].to_numpy(),
        paired["y_true_ablation"].to_numpy(),
    ):
        raise RuntimeError(
            f"Labels differ between CTE-Net and {ablation_name}."
        )

    labels = paired["y_true_cte"].to_numpy(dtype=int)
    correct_cte = paired["y_pred_cte"].to_numpy(dtype=int) == labels
    correct_ablation = (
        paired["y_pred_ablation"].to_numpy(dtype=int) == labels
    )

    n10 = int(np.sum(correct_cte & ~correct_ablation))
    n01 = int(np.sum(~correct_cte & correct_ablation))
    discordant = n10 + n01
    p_raw = (
        float(binomtest(n10, discordant, p=0.5).pvalue)
        if discordant > 0
        else 1.0
    )

    return {
        "comparison": f"CTE-Net vs. {ablation_name}",
        "ablation": ablation_name,
        "n_subjects": len(paired),
        "n10_cte_correct_ablation_incorrect": n10,
        "n01_cte_incorrect_ablation_correct": n01,
        "discordant_pairs": discordant,
        "p_exact_raw": p_raw,
    }


mcnemar_holm = pd.DataFrame(
    [
        exact_mcnemar_against_cte(participant_consensus, ablation_name)
        for ablation_name in ABLATIONS
    ]
)
mcnemar_holm["p_holm"] = holm_adjust(mcnemar_holm["p_exact_raw"])
mcnemar_holm["significant_holm_0_05"] = mcnemar_holm["p_holm"] < 0.05
mcnemar_holm["direction"] = np.where(
    mcnemar_holm["significant_holm_0_05"],
    np.where(
        mcnemar_holm["n10_cte_correct_ablation_incorrect"]
        > mcnemar_holm["n01_cte_incorrect_ablation_correct"],
        "CTE-Net > ablation",
        "CTE-Net < ablation",
    ),
    "No significant difference",
)
mcnemar_holm["correction_family"] = "Holm across 2 ablation comparisons"
mcnemar_holm["analysis_unit"] = "participant"

mcnemar_holm.to_csv(
    OUTPUT_ROOT / "Ablation_McNemar_exact_Holm.csv",
    index=False,
)

print("Primary participant-level exact McNemar tests.")
display(mcnemar_holm)

Primary participant-level exact McNemar tests.


,comparison,ablation,n_subjects,n10_cte_correct_ablation_incorrect,n01_cte_incorrect_ablation_correct,discordant_pairs,p_exact_raw,p_holm,significant_holm_0_05,direction,correction_family,analysis_unit
0,CTE-Net vs. Without TE Module,Without TE Module,120,6,2,8,0.289062,0.289062,False,No significant difference,Holm across 2 ablation comparisons,participant
1,CTE-Net vs. Without Transformer,Without Transformer,120,17,5,22,0.016901,0.033801,True,CTE-Net > ablation,Holm across 2 ablation comparisons,participant


In [16]:
# ============================================================
# 8. COMPACT MANUSCRIPT-READY STATISTICAL SUMMARY
# ============================================================

manuscript_seed = wilcoxon_holm[
    [
        "metric",
        "ablation",
        "cte_net_mean_percent",
        "ablation_mean_percent",
        "mean_difference_cte_minus_ablation_pp",
        "p_raw",
        "p_holm",
        "direction",
    ]
].copy()

manuscript_participant = mcnemar_holm[
    [
        "ablation",
        "n10_cte_correct_ablation_incorrect",
        "n01_cte_incorrect_ablation_correct",
        "p_exact_raw",
        "p_holm",
        "direction",
    ]
].copy()

manuscript_seed.to_csv(
    OUTPUT_ROOT / "Ablation_seed_comparisons_for_manuscript.csv",
    index=False,
)
manuscript_participant.to_csv(
    OUTPUT_ROOT / "Ablation_participant_comparisons_for_manuscript.csv",
    index=False,
)

print("=" * 88)
print("ABLATION STATISTICAL ANALYSIS COMPLETED")
print("=" * 88)
print("\nSeed-level Wilcoxon-Holm results:")
display(manuscript_seed)
print("\nParticipant-level exact McNemar-Holm results:")
display(manuscript_participant)

print("\nFiles saved in:", OUTPUT_ROOT)
for path in sorted(OUTPUT_ROOT.glob("*")):
    print("-", path.name)

ABLATION STATISTICAL ANALYSIS COMPLETED

Seed-level Wilcoxon-Holm results:


,metric,ablation,cte_net_mean_percent,ablation_mean_percent,mean_difference_cte_minus_ablation_pp,p_raw,p_holm,direction
0,accuracy,Without TE Module,80.908972,78.657716,2.251256,0.013672,0.041016,CTE-Net > ablation
1,accuracy,Without Transformer,80.908972,73.393470,7.515502,0.001953,0.011719,CTE-Net > ablation
2,sensitivity,Without TE Module,84.186879,82.343345,1.843534,0.193359,0.193359,No significant difference
3,sensitivity,Without Transformer,84.186879,77.970961,6.215919,0.001953,0.011719,CTE-Net > ablation
4,precision,Without TE Module,82.659138,80.815095,1.844042,0.064453,0.128906,No significant difference
5,precision,Without Transformer,82.659138,75.691167,6.967971,0.001953,0.011719,CTE-Net > ablation



Participant-level exact McNemar-Holm results:


,ablation,n10_cte_correct_ablation_incorrect,n01_cte_incorrect_ablation_correct,p_exact_raw,p_holm,direction
0,Without TE Module,6,2,0.289062,0.289062,No significant difference
1,Without Transformer,17,5,0.016901,0.033801,CTE-Net > ablation



Files saved in: /kaggle/working/ablation_statistical_tests
- Ablation_McNemar_exact_Holm.csv
- Ablation_Wilcoxon_Holm_primary_metrics.csv
- Ablation_input_protocol_audit.csv
- Ablation_participant_comparisons_for_manuscript.csv
- Ablation_participant_consensus_predictions.csv
- Ablation_participant_metrics_by_seed.csv
- Ablation_participant_metrics_mean_std.csv
- Ablation_participant_predictions_by_seed.csv
- Ablation_seed_comparisons_for_manuscript.csv
- Ablation_window_metrics_by_fold_recomputed.csv
- Ablation_window_metrics_by_seed_recomputed.csv
- Ablation_window_metrics_mean_std.csv


## Interpretation guide

- Use the **Wilcoxon–Holm** results only to describe whether differences were consistent across the ten repeated model fits. They do not establish clinical superiority because the seeds are not independent participants.
- Use the **exact McNemar–Holm** results as the primary paired comparison of participant-level classification decisions.
- Do not infer significance from the mean values or overlapping confidence intervals alone.
- Report the Holm correction family explicitly: six prespecified seed-level tests and two participant-level ablation comparisons.